# Data Analysis & Cleaning: Atlantic Books Trending Dataset

This notebook performs end-to-end data cleaning, attribute extraction, and preprocessing on raw scraped book data (`books.csv`).

### Cleaning Workflow:
1. **Initial Inspection**: Load dataset and inspect structure.
2. **Column Pruning**: Remove scrape-specific navigation crumbs and unneeded URLs.
3. **Column Re-alignment**: Fix row shifts in bullet columns caused by review count badges.
4. **Metadata Extraction**: Parse author, publisher, and publisher imprint from consolidated text.
5. **Shift Correction**: Fix entries where publisher was mistakenly placed in author field.
6. **Classification Parsing**: Extract Subject and BISAC categories using regex boundary detection.
7. **Numeric Cleaning**: Clean prices, convert currency, and cast numeric fields.
8. **Missing Value Imputation & Standardization**: Handle nulls and standardize text casing.
9. **Exploratory Checks**: Inspect summary statistics and top records.
10. **Data Export**: Save cleaned public dataset to `atlantic_books_public.csv`.


## 1. Imports & Data Ingestion
Load core data manipulation libraries, regular expressions, and the raw scraped dataset.


In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load raw scraped book data
df = pd.read_csv('books.csv')
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


Dataset loaded: 945 rows, 21 columns


,page,title,writer,format,actual_price,price_before_discount,discount_percent,stars,url,bullet_1,...,bullet_3,bullet_4,bullet_5,bullet_6,bullet_7,bullet_8,bullet_9,bullet_10,bullet_11,bullet_12
0,1,Atomic Habits (Lead Title),"Clear, James",Paperback,₹650,₹999,35.0,3.5,https://atlanticbooks.com/collections/bestsell...,Current price,...,Original price,Original price,(-35%),Current price,|,/,(2),• Author(s): Clear | James • Publisher: Pengui...,Quick shop,NaN
1,1,Ikigai,Hector Garcia,Hardcover,₹390,₹599,35.0,NaN,https://atlanticbooks.com/collections/bestsell...,Current price,...,Original price,Original price,(-35%),Current price,|,/,• Author(s): Hector Garcia • Publisher: Pengui...,Quick shop,NaN,NaN
2,1,The Courage To Be Disliked How To Free Yoursel...,Ichiro Kishimi,Hardcover,₹390,₹599,35.0,5.0,https://atlanticbooks.com/collections/bestsell...,Current price,...,Original price,Original price,(-35%),Current price,|,/,(1),• Author(s): Ichiro Kishimi • Publisher: GROVE...,View full details,Quick shop
3,1,12 Years: My Messed-up Love Story,Chetan Bhagat,Paperback,₹260,₹399,35.0,NaN,https://atlanticbooks.com/collections/bestsell...,Current price,...,Original price,Original price,(-35%),Current price,|,/,• Author(s): Chetan Bhagat • Publisher: Harper...,View full details,Quick shop,NaN
4,1,The Mountain Is You: Transforming Self-Sabotag...,Brianna Wiest,Paperback,₹260,₹399,35.0,NaN,https://atlanticbooks.com/collections/bestsell...,Current price,...,Original price,Original price,(-35%),Current price,|,/,• Author(s): Brianna Wiest • Publisher: Manjul...,View full details,Quick shop,NaN


## 2. Drop Irrelevant Web Scraping Artifacts
Remove `url` and raw navigation tags (`bullet_1` to `bullet_8`). These columns only contain frontend UI labels (e.g. "Current price", "Original price", "|") scraped from the website cards.


In [2]:
# Drop URL and frontend UI button/label bullets
drop_crumbs = [
    'url',
    'bullet_1', 'bullet_2', 'bullet_3', 'bullet_4',
    'bullet_5', 'bullet_6', 'bullet_7', 'bullet_8'
]
df.drop(columns=drop_crumbs, inplace=True)
print("Shape after dropping UI artifacts:", df.shape)
df.head()


Shape after dropping UI artifacts: (945, 12)


,page,title,writer,format,actual_price,price_before_discount,discount_percent,stars,bullet_9,bullet_10,bullet_11,bullet_12
0,1,Atomic Habits (Lead Title),"Clear, James",Paperback,₹650,₹999,35.0,3.5,(2),• Author(s): Clear | James • Publisher: Pengui...,Quick shop,NaN
1,1,Ikigai,Hector Garcia,Hardcover,₹390,₹599,35.0,NaN,• Author(s): Hector Garcia • Publisher: Pengui...,Quick shop,NaN,NaN
2,1,The Courage To Be Disliked How To Free Yoursel...,Ichiro Kishimi,Hardcover,₹390,₹599,35.0,5.0,(1),• Author(s): Ichiro Kishimi • Publisher: GROVE...,View full details,Quick shop
3,1,12 Years: My Messed-up Love Story,Chetan Bhagat,Paperback,₹260,₹399,35.0,NaN,• Author(s): Chetan Bhagat • Publisher: Harper...,View full details,Quick shop,NaN
4,1,The Mountain Is You: Transforming Self-Sabotag...,Brianna Wiest,Paperback,₹260,₹399,35.0,NaN,• Author(s): Brianna Wiest • Publisher: Manjul...,View full details,Quick shop,NaN


## 3. Resolve Column Alignment & Shifts
During card scraping, items with customer review counts (e.g. `(1)`, `(2)`) pushed the actual metadata into `bullet_10`, `bullet_11`, and `bullet_12`.

We shift this metadata back into `bullet_9` so all book details reside in a single column, then drop the excess bullet columns.


In [3]:
# Fix 2-column shifts where bullet_12 contains data
mask_12 = df['bullet_12'].notna()
df.loc[mask_12, 'bullet_9'] = df.loc[mask_12, 'bullet_10']
df.loc[mask_12, 'bullet_10'] = df.loc[mask_12, 'bullet_11']
df.loc[mask_12, 'bullet_11'] = df.loc[mask_12, 'bullet_12']
df.loc[mask_12, 'bullet_12'] = None
df.drop(columns=['bullet_12', 'bullet_11'], inplace=True)

# Fix 1-column shifts where author metadata ended up in bullet_10
mask_10 = df['bullet_10'].str.contains('author', case=False, na=False)
df.loc[mask_10, 'bullet_9'] = df.loc[mask_10, 'bullet_10']
df.loc[mask_10, 'bullet_10'] = None
df.drop(columns=['bullet_10'], inplace=True)

print("Shape after column realignment:", df.shape)
df.head()


Shape after column realignment: (945, 9)


,page,title,writer,format,actual_price,price_before_discount,discount_percent,stars,bullet_9
0,1,Atomic Habits (Lead Title),"Clear, James",Paperback,₹650,₹999,35.0,3.5,• Author(s): Clear | James • Publisher: Pengui...
1,1,Ikigai,Hector Garcia,Hardcover,₹390,₹599,35.0,NaN,• Author(s): Hector Garcia • Publisher: Pengui...
2,1,The Courage To Be Disliked How To Free Yoursel...,Ichiro Kishimi,Hardcover,₹390,₹599,35.0,5.0,• Author(s): Ichiro Kishimi • Publisher: GROVE...
3,1,12 Years: My Messed-up Love Story,Chetan Bhagat,Paperback,₹260,₹399,35.0,NaN,• Author(s): Chetan Bhagat • Publisher: Harper...
4,1,The Mountain Is You: Transforming Self-Sabotag...,Brianna Wiest,Paperback,₹260,₹399,35.0,NaN,• Author(s): Brianna Wiest • Publisher: Manjul...


## 4. Extract Structured Attributes from Bullet 9
`bullet_9` contains metadata separated by bullet points (`•`). We parse this string to extract `author`, `publisher`, and `publisher_imprint`.


In [4]:
# Split metadata by the bullet delimiter
parts = df['bullet_9'].str.split('•')

df['author'] = parts.str[1]
df['publisher'] = parts.str[2]
df['publisher_imprint'] = parts.str[3]

# Strip standard label prefixes
df['author'] = df['author'].str.replace('Author(s): ', '', regex=False)
df['publisher'] = df['publisher'].str.replace('Publisher: ', '', regex=False)
df['publisher_imprint'] = df['publisher_imprint'].str.replace('Publisher Imprint: ', '', regex=False)

df[['title', 'author', 'publisher', 'publisher_imprint']].head()


,title,author,publisher,publisher_imprint
0,Atomic Habits (Lead Title),Clear | James,Penguin,Penguin Random House
1,Ikigai,Hector Garcia,Penguin,Penguin Random House
2,The Courage To Be Disliked How To Free Yoursel...,Ichiro Kishimi,GROVE ATLANTIC,Allen & Unwin
3,12 Years: My Messed-up Love Story,Chetan Bhagat,HarperCollins Publishers India,HarperCollins Publishers India
4,The Mountain Is You: Transforming Self-Sabotag...,Brianna Wiest,Manjul Publishing,Amaryllis


## 5. Correct Misaligned Author and Publisher Fields
Some records in `bullet_9` do not have an author tag, starting directly with the publisher. In those cases, the publisher was placed into the `author` column and imprint into `publisher`.

We identify these rows and shift values to their proper positions.


In [5]:
# Detect rows where publisher was mistakenly placed in author
mask_shifted = df['author'].str.contains('Publisher:', na=False)

# Shift imprint and publisher
df.loc[mask_shifted, 'publisher_imprint'] = df.loc[mask_shifted, 'publisher']
df.loc[mask_shifted, 'publisher'] = df.loc[mask_shifted, 'author']
df.loc[mask_shifted, 'author'] = None

# Re-clean stripped prefixes on shifted rows
df['publisher'] = df['publisher'].str.replace('Publisher: ', '', regex=False)
df['publisher_imprint'] = df['publisher_imprint'].str.replace('Publisher Imprint: ', '', regex=False)

# Verify no publisher tags remain in author column
remaining_issues = df['author'].str.contains('Publisher:', na=False).sum()
print("Remaining misplaced publisher rows:", remaining_issues)
df[['title', 'writer', 'publisher', 'publisher_imprint']].head()


Remaining misplaced publisher rows: 0


,title,writer,publisher,publisher_imprint
0,Atomic Habits (Lead Title),"Clear, James",Penguin,Penguin Random House
1,Ikigai,Hector Garcia,Penguin,Penguin Random House
2,The Courage To Be Disliked How To Free Yoursel...,Ichiro Kishimi,GROVE ATLANTIC,Allen & Unwin
3,12 Years: My Messed-up Love Story,Chetan Bhagat,HarperCollins Publishers India,HarperCollins Publishers India
4,The Mountain Is You: Transforming Self-Sabotag...,Brianna Wiest,Manjul Publishing,Amaryllis


## 6. Parse Subject & BISAC Categories (Regex Splitting)
In the raw data, the BISAC/Subject category name and the book synopsis are often glued together without a space (e.g. `Body Mind And SpiritThis is a book...`).

We use regex to identify lowercase-to-uppercase word transitions (`(?<=[a-z]{2})(?=[A-Z])`) while protecting legitimate CamelCase terms (e.g. `WordPress`, `JavaScript`).


In [6]:
# Regex boundary: at least 2 lowercase letters followed immediately by an uppercase letter
GLUE = re.compile(r'(?<=[a-z]{2})(?=[A-Z])')
PROTECTED = re.compile('WordPress|JavaScript|TypeScript|YouTube|PowerPoint|LinkedIn|PlayStation')

def _split_glued_text(text):
    """Separate category name from book description glued to it."""
    if not text:
        return np.nan, np.nan
    spans = [m.span() for m in PROTECTED.finditer(text)]
    m = next((m for m in GLUE.finditer(text) if not any(a < m.start() < b for a, b in spans)), None)
    val, desc = (text[:m.start()], text[m.start():]) if m else (text, '')
    return val.strip() or np.nan, desc.strip() or np.nan

def split_bullet_categories(dataframe, col='bullet_9'):
    """Parse subject and bisac key-value pairs from the bullet string."""
    def parse_entry(s):
        f = {k.lower(): v.strip() for k, v in re.findall(r'•\s*([^:•]+?)\s*:\s*([^•]*)', s)} if isinstance(s, str) else {}
        return [*_split_glued_text(f.get('subject')), *_split_glued_text(f.get('bisac'))]
    dataframe[['subject', 'subject_description', 'bisac', 'bisac_description']] = [parse_entry(s) for s in dataframe[col]]
    return dataframe

df = split_bullet_categories(df, col='bullet_9')
df[['subject', 'subject_description', 'bisac', 'bisac_description']].head()


,subject,subject_description,bisac,bisac_description
0,General Books,NaN,NaN,NaN
1,General Books,NaN,NaN,NaN
2,NaN,NaN,Personal Growth - Success,"Ichiro Kishimi lives in Kyoto. He writes, lect..."
3,NaN,NaN,General,From India's top-selling writer Chetan Bhagat ...
4,NaN,NaN,Body Mind And Spirit,This is a book about self-sabotage. Why we do ...


## 7. Clean Price & Currency Fields
- Drop rows missing benchmark price (`price_before_discount`).
- Strip currency symbols (`₹`) and thousands commas.
- Convert single USD price at index 844 (`$3`) to INR (~286).
- Cast both price fields to integer.


In [7]:
# Drop records without benchmark price
df.drop(df[df['price_before_discount'].isna()].index, inplace=True)

# Clean actual_price
df['actual_price'] = df['actual_price'].str.strip().str.replace('₹', '', regex=False).str.replace(',', '', regex=False)

# Convert single USD item to INR
if 844 in df.index and '$' in str(df.loc[844, 'actual_price']):
    df.loc[844, 'actual_price'] = str(round(3 * 95.18))

df['actual_price'] = df['actual_price'].astype(int)

# Clean price_before_discount
df['price_before_discount'] = (
    df['price_before_discount']
    .str.strip()
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(int)
)

print(f"Cleaned price range: ₹{df['actual_price'].min()} - ₹{df['actual_price'].max()}")
df[['actual_price', 'price_before_discount', 'discount_percent']].head()


Cleaned price range: ₹65 - ₹5916


,actual_price,price_before_discount,discount_percent
0,650,999,35.0
1,390,599,35.0
2,390,599,35.0
3,260,399,35.0
4,260,399,35.0


## 8. Missing Value Imputation & Text Standardization
- Drop the redundant temporary `author` column (`writer` already contains the authoritative author name from the product card).
- Standardize all text columns to lowercase and strip whitespace.
- Impute missing values in categorical columns with descriptive defaults (`unknown`, `unknown publisher`, `unknown publisher imprint`).


In [8]:
# Drop temporary author column
df.drop(columns=['author'], inplace=True)

# Standardize text formatting
text_cols = [
    'title', 'writer', 'format', 'publisher', 'publisher_imprint',
    'subject', 'subject_description', 'bisac', 'bisac_description'
]
df[text_cols] = df[text_cols].map(lambda x: x.strip().lower() if isinstance(x, str) else x)

# Impute categorical missing values
df['format'] = df['format'].fillna('unknown')
df['publisher'] = df['publisher'].fillna('unknown publisher')
df['publisher_imprint'] = df['publisher_imprint'].fillna('unknown publisher imprint')

df.head()


,page,title,writer,format,actual_price,price_before_discount,discount_percent,stars,bullet_9,publisher,publisher_imprint,subject,subject_description,bisac,bisac_description
0,1,atomic habits (lead title),"clear, james",paperback,650,999,35.0,3.5,• Author(s): Clear | James • Publisher: Pengui...,penguin,penguin random house,general books,NaN,NaN,NaN
1,1,ikigai,hector garcia,hardcover,390,599,35.0,NaN,• Author(s): Hector Garcia • Publisher: Pengui...,penguin,penguin random house,general books,NaN,NaN,NaN
2,1,the courage to be disliked how to free yoursel...,ichiro kishimi,hardcover,390,599,35.0,5.0,• Author(s): Ichiro Kishimi • Publisher: GROVE...,grove atlantic,allen & unwin,NaN,NaN,personal growth - success,"ichiro kishimi lives in kyoto. he writes, lect..."
3,1,12 years: my messed-up love story,chetan bhagat,paperback,260,399,35.0,NaN,• Author(s): Chetan Bhagat • Publisher: Harper...,harpercollins publishers india,harpercollins publishers india,NaN,NaN,general,from india's top-selling writer chetan bhagat ...
4,1,the mountain is you: transforming self-sabotag...,brianna wiest,paperback,260,399,35.0,NaN,• Author(s): Brianna Wiest • Publisher: Manjul...,manjul publishing,amaryllis,NaN,NaN,body mind and spirit,this is a book about self-sabotage. why we do ...


## 9. Summary Statistics & Sanity Checks
Inspect numeric summaries and check top-ranked items to ensure data integrity.


In [9]:
# Numeric summary of prices and discounts
df[['actual_price', 'price_before_discount', 'discount_percent']].describe()


,actual_price,price_before_discount,discount_percent
count,944.000000,944.000000,944.000000
mean,507.315678,770.468220,34.587924
std,444.144713,644.326945,3.795781
min,65.000000,99.000000,5.000000
25%,325.000000,499.000000,35.000000
50%,390.000000,599.000000,35.000000
75%,520.000000,799.000000,35.000000
max,5916.000000,7910.000000,43.000000


In [10]:
# Top 10 most expensive titles
df.nlargest(10, 'actual_price')[
    ['title', 'writer', 'actual_price', 'price_before_discount', 'discount_percent']
]


,title,writer,actual_price,price_before_discount,discount_percent
636,a game of thrones: the story continues,george r.r. martin,5916,7910,25.0
542,the chronicles of narnia box set,c. s. lewis,4731,6712,30.0
725,the mahabharata (box set),bibek debroy,3575,5500,35.0
874,vegetables: the indian way: a definitive colle...,camellia panjabi,3574,5200,31.0
813,berserk deluxe volume 1,kentaro miura,3445,5299,35.0
79,harry potter box set: the complete collection ...,j. k. rowling,3250,4999,35.0
333,diary of a wimpy kid boxset 15 books collection,jeff kinney,3250,4999,35.0
803,"5 lb. book of gre practice problems: 1,800+ pr...",manhattan prep,3157,3920,19.0
768,shatter me collection,tahereh mafi,2730,4199,35.0
346,alchemised: (english edition) the global fanta...,senlinyu,2688,2860,6.0


## 10. Export Clean Dataset
Remove intermediate parsing columns (`bullet_9`, `subject_description`, `bisac_description`) and export the final dataset to `atlantic_books_public.csv`.


In [11]:
# Drop raw extraction and intermediate columns
drop_cols = ['bullet_9', 'subject_description', 'bisac_description']
public_df = df.drop(columns=drop_cols, errors='ignore').copy().reset_index(drop=True)

# Export clean CSV
public_df.to_csv('atlantic_books_public.csv', index=False)

print(f"Exported clean dataset: {public_df.shape[0]} rows, {public_df.shape[1]} columns")
print("Columns:", public_df.columns.tolist())
public_df.info()


Exported clean dataset: 944 rows, 12 columns
Columns: ['page', 'title', 'writer', 'format', 'actual_price', 'price_before_discount', 'discount_percent', 'stars', 'publisher', 'publisher_imprint', 'subject', 'bisac']
<class 'pandas.DataFrame'>
RangeIndex: 944 entries, 0 to 943
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   page                   944 non-null    int64  
 1   title                  944 non-null    str    
 2   writer                 944 non-null    str    
 3   format                 944 non-null    str    
 4   actual_price           944 non-null    int64  
 5   price_before_discount  944 non-null    int64  
 6   discount_percent       944 non-null    float64
 7   stars                  14 non-null     float64
 8   publisher              944 non-null    str    
 9   publisher_imprint      944 non-null    str    
 10  subject                91 non-null     str    
 11  bisac    

In [12]:
# Sample records from the final dataset
public_df.head()


,page,title,writer,format,actual_price,price_before_discount,discount_percent,stars,publisher,publisher_imprint,subject,bisac
0,1,atomic habits (lead title),"clear, james",paperback,650,999,35.0,3.5,penguin,penguin random house,general books,NaN
1,1,ikigai,hector garcia,hardcover,390,599,35.0,NaN,penguin,penguin random house,general books,NaN
2,1,the courage to be disliked how to free yoursel...,ichiro kishimi,hardcover,390,599,35.0,5.0,grove atlantic,allen & unwin,NaN,personal growth - success
3,1,12 years: my messed-up love story,chetan bhagat,paperback,260,399,35.0,NaN,harpercollins publishers india,harpercollins publishers india,NaN,general
4,1,the mountain is you: transforming self-sabotag...,brianna wiest,paperback,260,399,35.0,NaN,manjul publishing,amaryllis,NaN,body mind and spirit
